In [1]:
"""
Этап 7. Список кандидатов на сокращение.

Начинаем с группы CZ (2426 SKU). Применяем дополнительные фильтры:
- возраст товара;
- клиентская база;
- повторные покупки;
- сезонность.

Цель — отобрать безопасные кандидаты для ~20% сокращения.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql, execute_sql, _get_conn_params

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

REPORTS_DIR = ROOT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

print("✅ Окружение готово")

✅ Окружение готово


In [2]:
"""
Фаза 3. Диагностика: сколько клиентов имеют несколько строк
одного SKU в одном счёте?
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql, execute_sql, _get_conn_params

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 1. Сколько пар (SKU, customer_id, invoice_no) имеют >1 строки?
multi_lines = read_sql("""
    SELECT
        COUNT(*) AS total_groups,
        COUNT(*) FILTER (WHERE n > 1) AS groups_with_multiple_lines,
        SUM(n) AS total_rows,
        SUM(n) FILTER (WHERE n > 1) AS rows_in_multi_groups
    FROM (
        SELECT
            stock_code,
            customer_id,
            invoice_no,
            COUNT(*) AS n
        FROM sales
        WHERE customer_id IS NOT NULL
        GROUP BY stock_code, customer_id, invoice_no
    ) t;
""")
print("Масштаб проблемы с повторными покупками:")
print(multi_lines.T)

# 2. Как это влияет на подсчёт repeat_customers?
# Старый метод (COUNT(*)) vs новый (COUNT(DISTINCT invoice_no))
impact = read_sql("""
    WITH cust_purchases_v1 AS (
        -- Старый способ: COUNT(*) — клиент считается повторным,
        -- если у него >1 СТРОКИ одного SKU
        SELECT
            stock_code,
            customer_id,
            COUNT(*) AS n
        FROM sales
        WHERE customer_id IS NOT NULL
        GROUP BY stock_code, customer_id
    ),
    cust_purchases_v2 AS (
        -- Новый способ: COUNT(DISTINCT invoice_no) — клиент считается
        -- повторным, если у него >1 РАЗНЫХ СЧЁТОВ
        SELECT
            stock_code,
            customer_id,
            COUNT(DISTINCT invoice_no) AS n
        FROM sales
        WHERE customer_id IS NOT NULL
        GROUP BY stock_code, customer_id
    )
    SELECT
        'v1 COUNT(*)' AS method,
        COUNT(DISTINCT stock_code) FILTER (WHERE n >= 2) AS skus_with_repeat,
        COUNT(*) FILTER (WHERE n >= 2) AS total_repeat_customers
    FROM cust_purchases_v1
    UNION ALL
    SELECT
        'v2 COUNT(DISTINCT invoice_no)' AS method,
        COUNT(DISTINCT stock_code) FILTER (WHERE n >= 2) AS skus_with_repeat,
        COUNT(*) FILTER (WHERE n >= 2) AS total_repeat_customers
    FROM cust_purchases_v2;
""")
print("\nСравнение методов:")
display(impact)

Масштаб проблемы с повторными покупками:
                                 0
total_groups                765998
groups_with_multiple_lines   33447
total_rows                  802551
rows_in_multi_groups         70000

Сравнение методов:


,method,skus_with_repeat,total_repeat_customers
0,v1 COUNT(*),3998,136941
1,v2 COUNT(DISTINCT invoice_no),3847,121311
